## Customer & Pricing Analytics: Telco Churn Model Stress Test

### Core question

> **Can a customer churn model remain stable when customer prices change?**

### Technical question

> **How sensitive are churn predictions to simulated changes in `MonthlyCharges`?**

### Experiment design

This notebook trains **one Logistic Regression baseline model**, freezes it, and then applies simulated pricing changes to the **test set only**:

- Normal: 0%
- Mild: +5%
- Moderate: +10%
- High: +15%
- Severe: +20%

The experiment measures **prediction sensitivity**, not actual causal churn.

> **Important:** The simulated price increases are assumptions for a stress test. They do not prove that a particular price increase causes customers to churn.


## 1. Hypothesis

> **Hypothesis:** Increasing simulated `MonthlyCharges` may increase the model's predicted probability of churn, but the magnitude of the change may differ across customers and customer segments.

This is deliberately a testable hypothesis rather than an assumption that the model must fail or that customers will actually churn.


## 2. Dataset

This notebook uses the **IBM Telco Customer Churn** dataset.

Place the CSV file in:

```text
data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
```

The main variables of interest are:

- `MonthlyCharges` — the variable being stress-tested
- `Churn` — the target
- `tenure` — customer tenure
- `Contract` — contract type
- `InternetService`
- `PaymentMethod`
- `PaperlessBilling`
- `SeniorCitizen`
- `Partner`
- `Dependents`

### Important methodological decision

`TotalCharges` is inspected for data quality but is **not used as a model feature**.

Why?

`TotalCharges` is cumulative, while the experiment changes current `MonthlyCharges`. Changing only monthly charges while keeping cumulative charges unchanged could create an internally inconsistent hypothetical customer record. Excluding it keeps the stress test more focused on current pricing.


## 3. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

## 4. Load the Dataset

In [ ]:
file_path = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

## 5. Initial Data Inspection

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
print(df.columns.tolist())

In [ ]:
df.info()

## 6. Data Quality Checks

Before modeling, check:

- Missing values
- Blank strings
- Data types
- Duplicate rows
- Duplicate customer IDs
- Target distribution

The goal is to understand the data without turning this into a large data-cleaning project.


In [ ]:
missing_values = df.isnull().sum()
print("Missing values:")
print(missing_values[missing_values > 0])

In [ ]:
blank_counts = (df == " ").sum()
print("Blank-string values:")
print(blank_counts[blank_counts > 0])

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

## 7. Convert `TotalCharges` to Numeric

Some versions of this dataset contain blank strings in `TotalCharges`.

We convert the column to numeric for inspection, but it will **not** be used in the final model.


In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("Missing TotalCharges after conversion:",
      df["TotalCharges"].isnull().sum())

## 8. Examine the Target Variable

`Churn` is the prediction target.

We will later encode:

- `No` → 0
- `Yes` → 1


In [ ]:
print(df["Churn"].value_counts())

In [ ]:
print(df["Churn"].value_counts(normalize=True) * 100)

## 9. Churn Distribution

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(data=df, x="Churn")

plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

## 10. Explore `MonthlyCharges`

`MonthlyCharges` is the variable that will be stress-tested.

First inspect its descriptive statistics and distribution.


In [ ]:
df["MonthlyCharges"].describe()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="MonthlyCharges",
    bins=30,
    kde=True
)

plt.title("Distribution of Monthly Charges")
plt.xlabel("Monthly Charges")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

## 11. Monthly Charges by Churn Status

This is **contextual analysis only**.

A difference between groups does not establish that price causes churn. It simply helps explain why `MonthlyCharges` is a reasonable variable to investigate.


In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=df,
    x="Churn",
    y="MonthlyCharges"
)

plt.title("Monthly Charges by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")

plt.tight_layout()
plt.show()

## 12. Encode the Target

In [ ]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print(df["Churn"].value_counts())

## 13. Select Modeling Features

We deliberately use a manageable feature set.

### Numerical features

- `tenure`
- `MonthlyCharges`
- `SeniorCitizen`

### Categorical features

- `Contract`
- `InternetService`
- `PaymentMethod`
- `PaperlessBilling`
- `Partner`
- `Dependents`

### Excluded

`TotalCharges` is excluded to avoid complicating the pricing stress scenario with cumulative-charge assumptions.


In [ ]:
features = [
    "tenure",
    "MonthlyCharges",
    "Contract",
    "InternetService",
    "PaymentMethod",
    "PaperlessBilling",
    "SeniorCitizen",
    "Partner",
    "Dependents"
]

target = "Churn"

X = df[features].copy()
y = df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

## 14. Train/Test Split

We use:

- 80% training data
- 20% testing data
- Stratification by `Churn`
- Fixed `random_state` for reproducibility

The stress scenarios will only modify the **test data**.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

In [ ]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

## 15. Define Preprocessing

Categorical variables need one-hot encoding.

The preprocessing is placed inside a `Pipeline` so that preprocessing is fitted using the training data only, helping prevent data leakage.


In [ ]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "SeniorCitizen"
]

categorical_features = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "PaperlessBilling",
    "Partner",
    "Dependents"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

## 16. Create the Baseline Model

We use **one Logistic Regression model**.

The objective is not to compare many algorithms. The model is simply the instrument used for the stress test.


In [ ]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", logistic_model)
    ]
)

print(model)

## 17. Train the Baseline Model

At this point, the model learns from the original historical training data.

After this step, the model will be **frozen**. We will not retrain it for the stress scenarios.


In [ ]:
model.fit(X_train, y_train)

print("Model trained successfully.")

## 18. Baseline Predictions

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Predictions generated.")

## 19. Baseline Model Performance

These metrics describe the model on the original test set.

They are **baseline model metrics**, not the results of the pricing stress test.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Baseline Model Performance")
print("--------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Churn", "Churn"]
))

## 20. Baseline Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.title("Baseline Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()

# 🔥 21. Begin the Stress Test

## Why we do not evaluate stressed accuracy/recall

The historical `Churn` label tells us what happened under the **original observed conditions**.

After we simulate a +20% price increase, we do **not** know what the customer's true churn outcome would have been.

Therefore, we should not pretend that the original `Churn` labels are the true outcomes under the hypothetical price increase.

Instead, the stress test focuses on:

- Average predicted churn probability
- Predicted churn rate
- Number of predictions that change
- Customers switching from predicted No Churn → predicted Churn
- Differences across predefined customer segments

This makes the experiment a **model sensitivity analysis**, not a causal pricing experiment.


## 22. Preserve the Original Test Set

In [ ]:
test_normal = X_test.copy()

print(test_normal.head())

## 23. Inspect the Training Price Range

At high stress levels, some simulated prices may fall outside the range the model observed during training.

This is important because predictions outside the historical range involve extrapolation concerns.


In [ ]:
train_min_charge = X_train["MonthlyCharges"].min()
train_max_charge = X_train["MonthlyCharges"].max()

test_min_charge = test_normal["MonthlyCharges"].min()
test_max_charge = test_normal["MonthlyCharges"].max()

print("Training MonthlyCharges range:")
print("Minimum:", train_min_charge)
print("Maximum:", train_max_charge)

print("\nTesting MonthlyCharges range:")
print("Minimum:", test_min_charge)
print("Maximum:", test_max_charge)

## 24. Define Stress Scenarios

These are **simulation assumptions**, not claims about normal industry pricing behavior.

| Scenario | Change in MonthlyCharges |
|---|---:|
| Normal | 0% |
| Mild | +5% |
| Moderate | +10% |
| High | +15% |
| Severe | +20% |


In [ ]:
stress_levels = {
    "Normal": 0.00,
    "+5%": 0.05,
    "+10%": 0.10,
    "+15%": 0.15,
    "+20%": 0.20
}

stress_levels

## 25. Create All Stress-Test Datasets

Only `MonthlyCharges` is modified.

All other customer characteristics remain unchanged.

The model itself is not retrained.


In [ ]:
stress_datasets = {}

for scenario, increase in stress_levels.items():

    stressed_data = test_normal.copy()

    stressed_data["MonthlyCharges"] = (
        stressed_data["MonthlyCharges"] * (1 + increase)
    )

    stress_datasets[scenario] = stressed_data

print("Stress datasets created:")
print(list(stress_datasets.keys()))

## 26. Check Monthly Charge Ranges Under Stress

In [ ]:
for scenario, data in stress_datasets.items():

    print(
        f"{scenario}: "
        f"min = {data['MonthlyCharges'].min():.2f}, "
        f"max = {data['MonthlyCharges'].max():.2f}"
    )

## 27. Check for Prices Outside the Training Range

In [ ]:
range_check = []

for scenario, data in stress_datasets.items():

    outside_count = (
        data["MonthlyCharges"] > train_max_charge
    ).sum()

    percentage = (
        outside_count / len(data)
    ) * 100

    range_check.append({
        "Scenario": scenario,
        "Maximum_MonthlyCharges": data["MonthlyCharges"].max(),
        "Outside_Training_Max": outside_count,
        "Percentage_Outside": percentage
    })

range_check_df = pd.DataFrame(range_check)

range_check_df

## 28. Run the Frozen Model Across All Scenarios

### Critical rule

**Do not call `.fit()` again.**

We use the same trained model for:

- Normal
- +5%
- +10%
- +15%
- +20%

This isolates the effect of the simulated pricing change on model predictions.


In [ ]:
predictions = {}

for scenario, data in stress_datasets.items():

    probabilities = model.predict_proba(data)[:, 1]
    classes = (probabilities >= 0.5).astype(int)

    predictions[scenario] = {
        "probabilities": probabilities,
        "classes": classes
    }

print("Stress testing completed.")

## 29. Inspect Example Predictions

In [ ]:
print("Normal probabilities:")
print(predictions["Normal"]["probabilities"][:10])

print("\n+20% probabilities:")
print(predictions["+20%"]["probabilities"][:10])

## 30. Average Predicted Churn Probability

This is one of the main stress-test measures.

It answers:

> **How does the model's average predicted churn probability change as the simulated price increases?**


In [ ]:
results = []

for scenario in stress_levels.keys():

    probabilities = predictions[scenario]["probabilities"]

    results.append({
        "Scenario": scenario,
        "Average_Predicted_Churn_Probability": probabilities.mean()
    })

probability_results = pd.DataFrame(results)

probability_results

## 31. Predicted Churn Rate

Using the same 0.5 classification threshold, calculate the percentage of customers the model predicts as churners under each scenario.

This is a **predicted churn rate**, not an observed or actual churn rate.


In [ ]:
churn_rate_results = []

for scenario in stress_levels.keys():

    classes = predictions[scenario]["classes"]

    predicted_churn_rate = classes.mean() * 100

    churn_rate_results.append({
        "Scenario": scenario,
        "Predicted_Churn_Rate_%": predicted_churn_rate
    })

churn_rate_df = pd.DataFrame(churn_rate_results)

churn_rate_df

## 32. Combine Main Stress-Test Results

In [ ]:
stress_results = probability_results.merge(
    churn_rate_df,
    on="Scenario"
)

stress_results

## 33. Change from the Normal Scenario

In [ ]:
normal_probability = (
    probability_results.loc[
        probability_results["Scenario"] == "Normal",
        "Average_Predicted_Churn_Probability"
    ].iloc[0]
)

stress_results["Change_From_Normal"] = (
    stress_results["Average_Predicted_Churn_Probability"]
    - normal_probability
)

stress_results

In [ ]:
stress_results["Percentage_Change_From_Normal"] = (
    stress_results["Change_From_Normal"]
    / normal_probability
) * 100

stress_results

## 34. How Many Predictions Changed?

Compare each stressed scenario with the original Normal prediction.

This measures **classification sensitivity**.


In [ ]:
normal_classes = predictions["Normal"]["classes"]

prediction_change_results = []

for scenario in stress_levels.keys():

    scenario_classes = predictions[scenario]["classes"]

    changed = (
        scenario_classes != normal_classes
    )

    number_changed = changed.sum()

    percentage_changed = (
        number_changed / len(normal_classes)
    ) * 100

    prediction_change_results.append({
        "Scenario": scenario,
        "Customers_Changed": number_changed,
        "Percentage_Changed": percentage_changed
    })

prediction_change_df = pd.DataFrame(
    prediction_change_results
)

prediction_change_df

## 35. No Churn → Churn Switches

This focuses specifically on customers whose predicted class changes from:

**No Churn → Churn**

under each simulated pricing scenario.


In [ ]:
switch_results = []

for scenario in stress_levels.keys():

    scenario_classes = predictions[scenario]["classes"]

    switched_to_churn = (
        (normal_classes == 0) &
        (scenario_classes == 1)
    )

    number_switched = switched_to_churn.sum()

    percentage_switched = (
        number_switched / len(normal_classes)
    ) * 100

    switch_results.append({
        "Scenario": scenario,
        "Switched_To_Churn": number_switched,
        "Percentage_Switched_To_Churn": percentage_switched
    })

switch_df = pd.DataFrame(switch_results)

switch_df

## 36. Individual Probability Changes

For every customer, calculate:

`Stressed predicted probability − Normal predicted probability`

This helps identify whether the model response is uniform or varies substantially across customers.


In [ ]:
probability_changes = {}

for scenario in stress_levels.keys():

    probability_changes[scenario] = (
        predictions[scenario]["probabilities"]
        - predictions["Normal"]["probabilities"]
    )

In [ ]:
mean_probability_changes = []

for scenario in stress_levels.keys():

    change = probability_changes[scenario]

    mean_probability_changes.append({
        "Scenario": scenario,
        "Mean_Probability_Change": change.mean(),
        "Median_Probability_Change": np.median(change),
        "Minimum_Change": change.min(),
        "Maximum_Change": change.max()
    })

probability_change_df = pd.DataFrame(
    mean_probability_changes
)

probability_change_df

# 📊 37. Main Visualization

## Model sensitivity to simulated pricing changes

This is the primary visual for the LinkedIn post.

The exact pattern must come from the model's actual results. Do not assume the relationship will be linear.


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    stress_results["Scenario"],
    stress_results["Average_Predicted_Churn_Probability"],
    marker="o",
    linewidth=2
)

plt.title(
    "Model Sensitivity to Simulated Monthly Charge Increases"
)

plt.xlabel("Pricing Stress Scenario")
plt.ylabel("Average Predicted Churn Probability")

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    stress_results["Scenario"],
    stress_results["Average_Predicted_Churn_Probability"],
    marker="o",
    linewidth=2
)

plt.title(
    "Model Sensitivity to Simulated Monthly Charge Increases"
)

plt.xlabel("Pricing Stress Scenario")
plt.ylabel("Average Predicted Churn Probability")

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    "../outputs/figures/stress_prediction_probability.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 📈 38. Prediction-Change Visualization

This shows what percentage of customer predictions changed relative to the original Normal scenario.


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    prediction_change_df["Scenario"],
    prediction_change_df["Percentage_Changed"],
    marker="o",
    linewidth=2
)

plt.title(
    "Percentage of Customer Predictions That Changed"
)

plt.xlabel("Pricing Stress Scenario")
plt.ylabel("Customers with Changed Prediction (%)")

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    prediction_change_df["Scenario"],
    prediction_change_df["Percentage_Changed"],
    marker="o",
    linewidth=2
)

plt.title(
    "Percentage of Customer Predictions That Changed"
)

plt.xlabel("Pricing Stress Scenario")
plt.ylabel("Customers with Changed Prediction (%)")

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    "../outputs/figures/prediction_changes.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 39. Optional Segment Analysis — Contract Type

We now examine whether the model's sensitivity differs across **predefined contract groups**:

- Month-to-month
- One year
- Two year

This is a model-sensitivity comparison.

It is **not** evidence that one contract group will actually churn more because of a price increase.


In [ ]:
segment_data = X_test.copy()

segment_data["Normal_Probability"] = (
    predictions["Normal"]["probabilities"]
)

segment_data["Stress_20_Probability"] = (
    predictions["+20%"]["probabilities"]
)

segment_data["Probability_Change"] = (
    segment_data["Stress_20_Probability"]
    - segment_data["Normal_Probability"]
)

segment_data.head()

In [ ]:
contract_results = (
    segment_data
    .groupby("Contract")["Probability_Change"]
    .agg(["mean", "median", "count"])
    .reset_index()
)

contract_results.columns = [
    "Contract",
    "Mean_Probability_Change",
    "Median_Probability_Change",
    "Customer_Count"
]

contract_results

## 40. Contract Sensitivity Visualization

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=contract_results,
    x="Contract",
    y="Mean_Probability_Change"
)

plt.title(
    "Change in Predicted Churn Probability by Contract Type"
)

plt.xlabel("Contract Type")
plt.ylabel("Mean Change in Predicted Churn Probability")

plt.xticks(rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=contract_results,
    x="Contract",
    y="Mean_Probability_Change"
)

plt.title(
    "Change in Predicted Churn Probability by Contract Type"
)

plt.xlabel("Contract Type")
plt.ylabel("Mean Change in Predicted Churn Probability")

plt.xticks(rotation=15)

plt.tight_layout()

plt.savefig(
    "../outputs/figures/contract_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 🧠 41. Critical Interpretation

Before writing the LinkedIn post, answer these questions using the **actual results**:

1. Did average predicted churn probability increase as `MonthlyCharges` increased?
2. Was the change gradual or nonlinear?
3. How many predictions changed under +20%?
4. How many switched from predicted No Churn → Churn?
5. Which contract segment showed the largest change?
6. Did any stress scenario move prices outside the model's historical training range?
7. Could the result reflect model sensitivity rather than actual customer behavior?
8. What was genuinely surprising about the result?

### Do not invent the answers.

The data and model output should determine the story.


# ⚠️ 42. Limitations

### 1. No causal inference

The experiment does not prove that increasing prices causes customers to churn.

### 2. Counterfactual problem

The dataset does not contain actual churn outcomes for these hypothetical pricing scenarios.

### 3. Simplified stress scenario

Only `MonthlyCharges` is changed while other customer characteristics are held constant.

### 4. `TotalCharges`

`TotalCharges` is excluded because changing current monthly charges without modeling the customer's full billing history could create an unrealistic counterfactual record.

### 5. Distribution shift

At higher stress levels, some simulated prices may exceed the range observed in the training data.

### 6. Dataset scope

The findings come from one historical telecom dataset and should not automatically be generalized to every telecom business or customer population.


# 💼 43. Business Interpretation

The appropriate interpretation is:

> A churn model can be sensitive to changes in pricing assumptions, but that sensitivity should not be interpreted as a causal estimate of customer behavior.

The practical lesson is broader:

> **Models should be stress-tested against plausible changes in the environment, not evaluated only on historical test accuracy.**


# 💾 44. Save Final Results

The following files will be saved:

- Stress-test results
- Contract sensitivity results
- Main stress-test figure
- Prediction-change figure
- Contract sensitivity figure


In [ ]:
from pathlib import Path

Path("../outputs/figures").mkdir(
    parents=True,
    exist_ok=True
)

Path("../outputs/results").mkdir(
    parents=True,
    exist_ok=True
)

final_results = (
    stress_results
    .merge(prediction_change_df, on="Scenario")
    .merge(switch_df, on="Scenario")
)

final_results_rounded = final_results.copy()

numeric_columns = (
    final_results_rounded
    .select_dtypes(include=np.number)
    .columns
)

final_results_rounded[numeric_columns] = (
    final_results_rounded[numeric_columns].round(4)
)

final_results_rounded.to_csv(
    "../outputs/results/stress_test_results.csv",
    index=False
)

contract_results.to_csv(
    "../outputs/results/contract_sensitivity_results.csv",
    index=False
)

range_check_df.to_csv(
    "../outputs/results/range_check.csv",
    index=False
)

print("Results saved successfully.")

# 45. Save the Trained Model

Saving the model is optional but useful for reproducibility.


In [ ]:
import joblib

joblib.dump(
    model,
    "../outputs/telco_churn_logistic_model.pkl"
)

print("Model saved successfully.")

# 📌 46. Print the Key Numbers for the LinkedIn Post

Run this cell after the experiment.

These are the numbers you should use when writing the post. **Do not replace them with invented example values.**


In [ ]:
normal_avg = stress_results.loc[
    stress_results["Scenario"] == "Normal",
    "Average_Predicted_Churn_Probability"
].iloc[0]

stress_avg = stress_results.loc[
    stress_results["Scenario"] == "+20%",
    "Average_Predicted_Churn_Probability"
].iloc[0]

normal_churn_rate = churn_rate_df.loc[
    churn_rate_df["Scenario"] == "Normal",
    "Predicted_Churn_Rate_%"
].iloc[0]

stress_churn_rate = churn_rate_df.loc[
    churn_rate_df["Scenario"] == "+20%",
    "Predicted_Churn_Rate_%"
].iloc[0]

changed_percentage = prediction_change_df.loc[
    prediction_change_df["Scenario"] == "+20%",
    "Percentage_Changed"
].iloc[0]

switched_percentage = switch_df.loc[
    switch_df["Scenario"] == "+20%",
    "Percentage_Switched_To_Churn"
].iloc[0]

print("========== KEY RESULTS ==========")

print(
    f"Normal average predicted churn probability: "
    f"{normal_avg:.4f}"
)

print(
    f"+20% average predicted churn probability: "
    f"{stress_avg:.4f}"
)

print(
    f"Normal predicted churn rate: "
    f"{normal_churn_rate:.2f}%"
)

print(
    f"+20% predicted churn rate: "
    f"{stress_churn_rate:.2f}%"
)

print(
    f"Predictions changed under +20%: "
    f"{changed_percentage:.2f}%"
)

print(
    f"Predictions switched to churn under +20%: "
    f"{switched_percentage:.2f}%"
)

# 47. Final Sanity Check

Before publishing anything, verify:

- One model was trained.
- The model was not retrained during stress testing.
- Only the test-set `MonthlyCharges` values were changed.
- The same 0.5 classification threshold was used.
- No stressed accuracy/recall/F1 was presented as if the hypothetical labels were known.
- Actual results, not assumed results, are used.
- Causal claims are avoided.


In [ ]:
print("========== FINAL CHECK ==========")
print("Dataset shape:", df.shape)
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Stress scenarios:", list(stress_levels.keys()))
print("Model:", type(logistic_model).__name__)
print("Model retrained during stress testing: NO")
print("Stress applied to: MonthlyCharges in test set only")
print("TotalCharges used in model: NO")
print("Experiment completed successfully.")

# 🔬 Week 2 — Final Takeaway

This experiment is intentionally small.

It does not attempt to build the most accurate churn model. Instead, it asks a different question:

> **How does an existing model behave when an important business condition changes?**

The key distinction is:

**Prediction sensitivity ≠ causal customer behavior.**

That distinction is the main critical-thinking contribution of this experiment.
